# Phase 4: ALS Recommendation Model

Train a Spark MLlib ALS (Alternating Least Squares) collaborative-filtering model on `gold/fact_ratings` and generate **top-10 movie recommendations per user**.

**Pipeline:**
1. Load & prepare rating data
2. Train/test split (80/20)
3. Baseline ALS model
4. Hyperparameter tuning via `TrainValidationSplit`
5. Evaluate best model (RMSE)
6. Generate & save top-10 recommendations per user

### 1. Setup & Load Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, count, avg, round as spark_round,
    row_number, desc,
)
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

spark = SparkSession.builder \
    .appName("ALS-Recommender") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

fact_ratings = spark.read.parquet("gold/fact_ratings")
print(f"fact_ratings: {fact_ratings.count():,} rows")
fact_ratings.printSchema()

### 2. Data Preparation & Train/Test Split

In [ ]:
als_data = fact_ratings.select(
    col("userId"),
    col("movieId"),
    col("rating").cast(FloatType()),
)

train, test = als_data.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

print(f"Train: {train.count():>12,}")
print(f"Test:  {test.count():>12,}")

### 3. Baseline ALS Model

In [ ]:
als = ALS(
    maxIter=10,
    rank=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42,
)

baseline_model = als.fit(train)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction",
)

baseline_preds = baseline_model.transform(test)
baseline_rmse = evaluator.evaluate(baseline_preds)
print(f"Baseline RMSE (rank=10, regParam=0.1): {baseline_rmse:.4f}")

### 4. Hyperparameter Tuning

Search over `rank` and `regParam` using `TrainValidationSplit` (single 80/20 hold-out — faster than k-fold CrossValidator on 32M rows).

In [ ]:
als_tuning = ALS(
    maxIter=10,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42,
)

param_grid = (
    ParamGridBuilder()
    .addGrid(als_tuning.rank, [10, 20])
    .addGrid(als_tuning.regParam, [0.05, 0.1, 0.2])
    .build()
)

tvs = TrainValidationSplit(
    estimator=als_tuning,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42,
)

print(f"Tuning {len(param_grid)} configurations...")
tvs_model = tvs.fit(train)

### 5. Best Model Evaluation

In [ ]:
best_model = tvs_model.bestModel

print(f"Best rank:     {best_model.rank}")
print(f"Best regParam: {best_model._java_obj.parent().getRegParam()}")

# RMSE on held-out test set
best_preds = best_model.transform(test)
best_rmse = evaluator.evaluate(best_preds)
print(f"\nBaseline RMSE: {baseline_rmse:.4f}")
print(f"Tuned RMSE:    {best_rmse:.4f}")

# All validation metrics
print("\n=== All Configurations ===")
for i, (params, metric) in enumerate(zip(param_grid, tvs_model.validationMetrics)):
    rank_val = params[als_tuning.rank]
    reg_val = params[als_tuning.regParam]
    print(f"  rank={rank_val:<4} regParam={reg_val:<5} RMSE={metric:.4f}")

### 6. Generate Top-10 Recommendations

In [ ]:
user_recs_raw = best_model.recommendForAllUsers(10)

# Explode into individual rows with rank
recs_flat = (
    user_recs_raw
    .select("userId", explode("recommendations").alias("rec"))
    .select(
        "userId",
        col("rec.movieId").alias("movieId"),
        spark_round(col("rec.rating"), 3).alias("predicted_rating"),
    )
)

# Add rank per user
w = Window.partitionBy("userId").orderBy(desc("predicted_rating"))
recs_ranked = recs_flat.withColumn("rank", row_number().over(w))

print(f"Total recommendation rows: {recs_ranked.count():,}")
print(f"Users with recommendations: {recs_ranked.select('userId').distinct().count():,}")

### 7. Enrich with Movie Titles & Save

In [ ]:
dim_movies = spark.read.parquet("gold/dim_movies_enriched")

recommendations = (
    recs_ranked
    .join(
        dim_movies.select("movieId", "title", "genres", "avg_rating"),
        "movieId",
        "left",
    )
    .select("userId", "rank", "movieId", "title", "genres",
            "predicted_rating", "avg_rating")
    .orderBy("userId", "rank")
)

recommendations.write.mode("overwrite").parquet("gold/recommendations")
print("gold/recommendations written.")

### 8. Save Model

In [ ]:
best_model.write().overwrite().save("models/als_best")
print("Model saved to models/als_best")

### 9. Sample Recommendations

In [ ]:
# Pick 3 sample users (a power user, a moderate user, a light user)
dim_users = spark.read.parquet("gold/dim_users")

sample_power = dim_users.filter(col("is_power_user")).select("userId").first()[0]
sample_moderate = dim_users.filter(
    (col("rating_count").between(50, 200)) & (~col("is_power_user"))
).select("userId").first()[0]
sample_light = dim_users.filter(col("rating_count") < 20).select("userId").first()[0]

sample_ids = [sample_power, sample_moderate, sample_light]
labels = ["Power User", "Moderate User", "Light User"]

recs = spark.read.parquet("gold/recommendations")

for uid, label in zip(sample_ids, labels):
    user_info = dim_users.filter(col("userId") == uid).first()
    print(f"\n=== {label} (userId={uid}, {user_info['rating_count']} ratings, "
          f"avg={user_info['avg_rating']}) ===")
    recs.filter(col("userId") == uid).show(10, truncate=50)

### 10. Summary

In [ ]:
n_users = recs.select("userId").distinct().count()
n_movies_recommended = recs.select("movieId").distinct().count()

print("=== Phase 4 Complete ===")
print(f"  Model:                ALS (rank={best_model.rank}, regParam={best_model._java_obj.parent().getRegParam()})")
print(f"  Test RMSE:            {best_rmse:.4f}")
print(f"  Users served:         {n_users:,}")
print(f"  Unique movies in recs:{n_movies_recommended:,}")
print(f"  Recs per user:        10")
print(f"\nOutputs:")
print(f"  gold/recommendations  — top-10 per user with titles")
print(f"  models/als_best       — serialized ALS model")

train.unpersist()
test.unpersist()